In [1]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Aotizhongxin_20130301-20170228.csv', encoding='latin-1')
df2 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Changping_20130301-20170228.csv', encoding='latin-1')
df3 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Dingling_20130301-20170228.csv', encoding='latin-1')
df4 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Dongsi_20130301-20170228.csv', encoding='latin-1')
df5 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Guanyuan_20130301-20170228.csv', encoding='latin-1')
df6 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Gucheng_20130301-20170228.csv', encoding='latin-1')
df7 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Huairou_20130301-20170228.csv', encoding='latin-1')
df8 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Nongzhanguan_20130301-20170228.csv', encoding='latin-1')
df9 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Shunyi_20130301-20170228.csv', encoding='latin-1')
df10 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Tiantan_20130301-20170228.csv', encoding='latin-1')
df11 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Wanliu_20130301-20170228.csv', encoding='latin-1')
df12 = pd.read_csv('/content/sample_data/Smart City Air Quality and Traffic/PRSA_Data_Wanshouxigong_20130301-20170228.csv', encoding='latin-1')

merged_df = pd.merge(df1, df2, how='outer')
merged_df = pd.merge(merged_df, df3, how='outer')
merged_df = pd.merge(merged_df, df4, how='outer')
merged_df = pd.merge(merged_df, df5, how='outer')
merged_df = pd.merge(merged_df, df6, how='outer')
merged_df = pd.merge(merged_df, df7, how='outer')
merged_df = pd.merge(merged_df, df8, how='outer')
merged_df = pd.merge(merged_df, df9, how='outer')
merged_df = pd.merge(merged_df, df10, how='outer')
merged_df = pd.merge(merged_df, df11, how='outer')
merged_df = pd.merge(merged_df, df12, how='outer')

In [2]:

# ════════════════════════════════════════════════════════════════
# STEP 1 — Load & Initial Inspection
# ════════════════════════════════════════════════════════════════

df = merged_df

print("=" * 55)
print("STEP 1 — Raw dataset")
print("=" * 55)
print(f"Shape           : {df.shape}")
print(f"Columns         : {df.columns.tolist()}")
print(f"\nNull % per column:\n{(df.isnull().mean() * 100).round(2).to_string()}")
print(f"\nSample:\n{df.head(3)}")


# ════════════════════════════════════════════════════════════════
# STEP 2 — Drop Unwanted Columns
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 2 — Drop unwanted columns")
print("=" * 55)

# Drop the row-number index column — carries no information
df.drop(columns=["No"], inplace=True)

# Drop any column with >50 % nulls
null_frac = df.isnull().mean()
cols_over_50 = null_frac[null_frac > 0.50].index.tolist()
if cols_over_50:
    print(f"Dropping >50 % null columns: {cols_over_50}")
    df.drop(columns=cols_over_50, inplace=True)
else:
    print("No columns exceed 50 % nulls — nothing extra dropped.")

# Columns we will actually use downstream
KEEP = ["datetime", "station", "PM2.5", "PM10", "SO2", "NO2",
        "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN",
        "wd_sin", "wd_cos", "WSPM"]

print(f"Remaining columns: {df.columns.tolist()}")


# ════════════════════════════════════════════════════════════════
# STEP 3 — Build datetime & Timezone Alignment
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 3 — Build datetime column")
print("=" * 55)

# The dataset splits date into year / month / day / hour columns.
# Combine them into a single datetime and treat everything as local
# time (Asia/Shanghai, UTC+8) — this is standard for PRSA Beijing data.
df["datetime"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]].rename(
        columns={"hour": "hour"}   # pd.to_datetime needs lowercase keys
    )
)

# If you ever receive a mixed-TZ version, normalise here:
# df["datetime"] = df["datetime"].dt.tz_localize("Asia/Shanghai")
# df["datetime"] = df["datetime"].dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)

df.drop(columns=["year", "month", "day", "hour"], inplace=True)

print(f"Datetime range  : {df['datetime'].min()}  →  {df['datetime'].max()}")
print(f"Total hours     : {len(df):,}")


# ════════════════════════════════════════════════════════════════
# STEP 4 — Handle Long NaN Runs (Sensor Outages)
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 4 — Flag & handle sensor-outage NaN runs")
print("=" * 55)

# Threshold: >= 3 consecutive missing rows = sensor outage
OUTAGE_THRESHOLD = 3

NUMERIC_COLS = [c for c in df.columns
                if c not in ("datetime", "station", "wd")
                and pd.api.types.is_numeric_dtype(df[c])]


def flag_outage_rows(series: pd.Series, threshold: int) -> pd.Series:
    """
    Returns a boolean Series that is True for every row that belongs
    to a consecutive-NaN run of length >= threshold.
    """
    is_null = series.isnull()
    # Each non-null value starts a new 'group'; cumsum counts groups
    group_id = (~is_null).cumsum()
    run_length = is_null.groupby(group_id).transform("sum")
    return is_null & (run_length >= threshold)


# Build a single outage mask across ALL numeric columns
outage_mask = pd.Series(False, index=df.index)
for col in NUMERIC_COLS:
    outage_mask |= flag_outage_rows(df[col], OUTAGE_THRESHOLD)

df["is_outage"] = outage_mask
print(f"Outage rows flagged : {df['is_outage'].sum():,}  "
      f"({df['is_outage'].mean() * 100:.2f} % of data)")

# ── Short gaps (1–2 consecutive NaNs): linear interpolation ──────
# Only interpolate rows NOT marked as outages
for col in NUMERIC_COLS:
    non_outage_idx = df.index[~df["is_outage"]]
    df.loc[non_outage_idx, col] = (
        df.loc[non_outage_idx, col]
          .interpolate(method="linear", limit=2, limit_direction="both")
    )

print(f"Short gaps interpolated (limit = 2 rows).")
print(f"Outage rows kept as NaN and flagged with is_outage = True.")


# ════════════════════════════════════════════════════════════════
# STEP 5 — Encode Wind Direction
# (This dataset uses compass strings like 'NW', 'ENE', not degrees)
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 5 — Encode wind direction (compass → sin / cos)")
print("=" * 55)

# Map compass abbreviations to degrees, then to sin/cos
COMPASS_TO_DEG = {
    "N": 0,   "NNE": 22.5,  "NE": 45,   "ENE": 67.5,
    "E": 90,  "ESE": 112.5, "SE": 135,  "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225,  "WSW": 247.5,
    "W": 270, "WNW": 292.5, "NW": 315,  "NNW": 337.5,
}

df["wd_deg"] = df["wd"].map(COMPASS_TO_DEG)          # NaN where wd is null
wd_rad       = np.deg2rad(df["wd_deg"])
df["wd_sin"] = np.sin(wd_rad)
df["wd_cos"] = np.cos(wd_rad)

df.drop(columns=["wd", "wd_deg"], inplace=True)

print(f"'wd' column replaced by wd_sin and wd_cos.")
print(f"Null wd_sin rows : {df['wd_sin'].isnull().sum():,}")


# ════════════════════════════════════════════════════════════════
# STEP 6 — Handle Remaining Nulls
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 6 — Handle remaining nulls")
print("=" * 55)

# Drop rows where the target (PM2.5) is still null
before = len(df)
df.dropna(subset=["PM2.5"], inplace=True)
print(f"Dropped {before - len(df):,} rows with null PM2.5 (target column).")

# Final ffill → bfill pass for isolated remaining nulls in all numeric cols
all_numeric = [c for c in df.columns
               if c not in ("datetime", "station", "wd", "is_outage")
               and pd.api.types.is_numeric_dtype(df[c])]

df[all_numeric] = df[all_numeric].ffill(limit=2).bfill(limit=2)

remaining_nulls = df[all_numeric].isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if remaining_nulls.empty:
    print("No nulls remain in numeric columns.")
else:
    print(f"Remaining nulls (inside outage blocks — expected):\n{remaining_nulls}")


# ════════════════════════════════════════════════════════════════
# STEP 7 — Remove / Cap Physical Outliers
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 7 — Remove out-of-range values")
print("=" * 55)

# Physical plausibility bounds for this dataset's units
BOUNDS = {
    "PM2.5": (0,   999),
    "PM10":  (0,  1500),
    "SO2":   (0,  2000),    # µg/m³
    "NO2":   (0,   500),    # µg/m³
    "CO":    (0, 50000),    # µg/m³  (dataset uses µg/m³, not ppm)
    "O3":    (0,   500),    # µg/m³
    "TEMP":  (-40,  60),    # °C
    "PRES":  (870, 1084),   # hPa
    "DEWP":  (-60,  35),    # °C
    "RAIN":  (0,   300),    # mm/h
    "WSPM":  (0,    75),    # m/s
}

for col, (lo, hi) in BOUNDS.items():
    if col not in df.columns:
        continue
    mask = (df[col] < lo) | (df[col] > hi)
    count = mask.sum()
    if count:
        print(f"  {col:8s}: {count:,} out-of-range → set NaN")
        df.loc[mask, col] = np.nan

# Re-interpolate the newly created NaNs
df[all_numeric] = df[all_numeric].interpolate(method="linear", limit=2)
print("Re-interpolated newly NaN'd outlier values (limit = 2 rows).")


# ════════════════════════════════════════════════════════════════
# STEP 8 — Align to a Regular Hourly Time Index
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 8 — Resample to gapless hourly index")
print("=" * 55)

df = df.sort_values("datetime").set_index("datetime")

# Resample numeric columns to hourly mean; keep is_outage as max
# (if any reading in that hour was an outage, mark the hour as outage)
agg_dict  = {c: "mean" for c in all_numeric if c in df.columns}
agg_dict["is_outage"] = "max"

station_frames = []
for station_name, grp in df.groupby("station"):
    resampled = grp.resample("1h").agg(agg_dict)
    resampled["station"] = station_name
    station_frames.append(resampled)

df = pd.concat(station_frames).reset_index()

print(f"Shape after resampling : {df.shape}")
print(f"Rows per station:\n{df.groupby('station').size().to_string()}")


# ════════════════════════════════════════════════════════════════
# STEP 9 — Create Lag Features
# (No separate traffic_count here; use CO and NO2 as traffic proxies)
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 9 — Create lag features")
print("=" * 55)

# The original dataset has no dedicated traffic_count column.
# CO and NO2 are the best available traffic-emission proxies.
LAG_PROXY_COLS = ["CO", "NO2"]
LAG_MINUTES    = [0, 60, 120, 180, 240, 300, 360]   # hourly data → 1 row = 60 min

df = df.sort_values(["station", "datetime"])

for col in LAG_PROXY_COLS:
    for lag_min in LAG_MINUTES:
        lag_rows = lag_min // 60          # 1 row = 1 hour
        new_col  = f"{col}_lag_{lag_min}min"
        df[new_col] = (
            df.groupby("station")[col]
              .shift(lag_rows)
        )

lag_cols = [c for c in df.columns if "_lag_" in c]
print(f"Lag columns created : {lag_cols}")


# ════════════════════════════════════════════════════════════════
# STEP 10 — Correlation Analysis: Best Lag for PM2.5 Prediction
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 10 — Correlation analysis (best predictor & lag)")
print("=" * 55)

results = []
for station, grp in df.groupby("station"):
    for lag_col in lag_cols:
        valid = grp[["PM2.5", lag_col]].dropna()
        if len(valid) < 30:
            continue
        corr = valid["PM2.5"].corr(valid[lag_col], method="pearson")
        proxy, lag_str = lag_col.rsplit("_lag_", 1)
        lag_min = int(lag_str.replace("min", ""))
        results.append({
            "station":    station,
            "proxy_col":  proxy,
            "lag_minutes": lag_min,
            "pearson_r":  corr,
        })

results_df = pd.DataFrame(results)
print(results_df.sort_values("pearson_r", ascending=False).to_string(index=False))

best = results_df.loc[results_df["pearson_r"].abs().idxmax()]
print(f"\n── Best predictor ──────────────────────────")
print(f"  Station     : {best['station']}")
print(f"  Proxy column: {best['proxy_col']}")
print(f"  Lag         : {best['lag_minutes']} minutes")
print(f"  Pearson r   : {best['pearson_r']:.4f}")


# ════════════════════════════════════════════════════════════════
# STEP 11 — Final Checks & Save
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 55)
print("STEP 11 — Final checks & save")
print("=" * 55)

print(f"Final shape     : {df.shape}")

remaining = df[all_numeric].isnull().sum()
remaining = remaining[remaining > 0]
if remaining.empty:
    print("No nulls remain in feature columns (outside flagged outage rows).")
else:
    print(f"Remaining nulls:\n{remaining}")

# df.to_parquet("cleaned_Gucheng_pm25.parquet", index=False)  # requires pyarrow
df.to_csv("67_Smart_City_Air_Quality_And_Traffic_Sample.csv", index=False)
print("Saved → cleaned_Gucheng_pm25.parquet")
print("Saved → cleaned_Gucheng_pm25.csv")


# ════════════════════════════════════════════════════════════════
# DELIVERABLE PARAGRAPH
# ════════════════════════════════════════════════════════════════

para = (
    f"Analysis of hourly air-quality data from the Gucheng monitoring station "
    f"(March 2013 – February 2017, {len(df):,} hourly records after cleaning) "
    f"shows that {best['proxy_col']} concentrations — a reliable proxy for "
    f"road-traffic emissions — are the strongest predictor of PM2.5 levels, "
    f"with a Pearson correlation of r = {best['pearson_r']:.3f} at a lag of "
    f"{best['lag_minutes']} minutes. This indicates that combustion-related "
    f"pollutants from traffic take approximately {best['lag_minutes']} minutes "
    f"to accumulate and register at the PM2.5 sensor, suggesting that real-time "
    f"{best['proxy_col']} readings could serve as an early-warning indicator for "
    f"deteriorating particulate air quality at this station."
)

print("\n── Deliverable paragraph ────────────────────────────\n")
print(para)

STEP 1 — Raw dataset
Shape           : (420768, 18)
Columns         : ['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']

Null % per column:
No         0.00
year       0.00
month      0.00
day        0.00
hour       0.00
PM2.5      2.08
PM10       1.53
SO2        2.14
NO2        2.88
CO         4.92
O3         3.16
TEMP       0.09
PRES       0.09
DEWP       0.10
RAIN       0.09
wd         0.43
WSPM       0.08
station    0.00

Sample:
   No  year  month  day  hour  PM2.5  PM10   SO2  NO2     CO    O3  TEMP  \
0   1  2013      3    1     0    3.0   6.0   3.0  8.0  300.0  44.0  -0.9   
1   1  2013      3    1     0    3.0   6.0  13.0  7.0  300.0  85.0  -2.3   
2   1  2013      3    1     0    4.0   4.0   3.0  NaN  200.0  82.0  -2.3   

     PRES  DEWP  RAIN  wd  WSPM    station  
0  1025.8 -20.5   0.0  NW   9.3     Shunyi  
1  1020.8 -19.7   0.0   E   0.5  Changping  
2  1020.8 -19.7   0.0   E   0.5   